# sub1quant release pipeline: package artifacts, publish to Hugging Face and GitHub

This notebook is a release path for the artifacts that already exist in this repository under `quantized/`. It runs in a local checkout or in Colab when the checkout and `quantized/` directory are available on the runtime.

The release flow does four concrete things:

1. Locates the project root and the real quantized artifacts.
2. Builds an inventory with file sizes, SHA-256 checksums, method names, status, and reported metrics from `README.md` and `EVAL_RESULTS.md`.
3. Stages selected artifacts into a release folder with `release_manifest.json`, `README.md`, `.gitattributes`, `checksums.sha256`, and `github_release_notes.md`.
4. Uploads the staged folder to Hugging Face Hub and the same files as GitHub Release assets when the run flags and tokens are set.

By default the release includes the non-broken quantized artifacts and records the failed SVD artifacts as skipped inventory. Set `INCLUDE_FAILED_SVD_ARTIFACTS=1` only when you intentionally want to publish the broken SVD files with their failure status clearly marked.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'huggingface_hub': 'huggingface_hub',
    'requests': 'requests',
    'dotenv': 'python-dotenv',
}

missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *missing])
else:
    print('Release dependencies are already available.')


## Runtime configuration

Local runs use the current checkout. Colab runs mount Google Drive and then search common locations for a checkout containing `quantized/` and `README.md`. If Colab cannot find the checkout, upload or mount this repository with the `quantized/` folder before continuing.

Release switches are environment variables so the same notebook can be rerun without editing cells:

- `RELEASE_TAG`: immutable tag for this release. Default: `sub1quant-gemma-4-e2b-ternary-aggressive-v1`
- `INCLUDE_EXPERIMENTAL_ARTIFACTS`: include `EXPERIMENTAL` artifacts. Default: `1`
- `INCLUDE_FAILED_SVD_ARTIFACTS`: include `BROKEN` SVD artifacts. Default: `0`
- `PACKAGING_MODE`: `hardlink` or `copy`. Default: `hardlink`
- `RUN_HF_UPLOAD`: upload staged release folder to Hugging Face. Default: `0`
- `RUN_GITHUB_RELEASE`: upload staged files as GitHub Release assets. Default: `0`


In [ ]:
from __future__ import annotations

import os
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)


def truthy(value: str | None, default: bool = False) -> bool:
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}


def run_command(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print('$ ' + ' '.join(args))
    return subprocess.run(args, cwd=str(cwd) if cwd else None, check=check, text=True, capture_output=True)


def find_project_root() -> Path | None:
    starts = [Path.cwd()]
    if IS_COLAB:
        starts.extend([
            Path('/content/sub1quant'),
            Path('/content/Quantization-Exploration'),
            Path('/content/drive/MyDrive/sub1quant'),
            Path('/content/drive/MyDrive/Quantization-Exploration'),
            Path('/content/drive/MyDrive/colab_experiments/sub1quant'),
        ])

    seen: set[Path] = set()
    for start in starts:
        try:
            start = start.resolve()
        except OSError:
            continue
        if start in seen:
            continue
        seen.add(start)
        candidates = [start, *start.parents]
        for candidate in candidates:
            if (candidate / 'quantized').is_dir() and (candidate / 'README.md').is_file():
                return candidate
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find a checkout with quantized/ and README.md. '
        'In Colab, mount or upload this repository including the quantized artifacts first.'
    )

QUANTIZED_DIR = PROJECT_ROOT / 'quantized'
DEFAULT_RELEASE_ROOT = Path('/content/drive/MyDrive/colab_experiments/sub1quant/releases') if IS_COLAB else PROJECT_ROOT / 'release'
RELEASE_ROOT = Path(os.getenv('RELEASE_ROOT', str(DEFAULT_RELEASE_ROOT))).expanduser().resolve()
RELEASE_NAME = os.getenv('RELEASE_NAME', 'sub1quant-gemma-4-e2b')
RELEASE_TAG = os.getenv('RELEASE_TAG', 'sub1quant-gemma-4-e2b-ternary-aggressive-v1')
INCLUDE_EXPERIMENTAL_ARTIFACTS = truthy(os.getenv('INCLUDE_EXPERIMENTAL_ARTIFACTS'), default=True)
INCLUDE_FAILED_SVD_ARTIFACTS = truthy(os.getenv('INCLUDE_FAILED_SVD_ARTIFACTS'), default=False)
PACKAGING_MODE = os.getenv('PACKAGING_MODE', 'hardlink').strip().lower()
CLEAN_RELEASE_DIR = truthy(os.getenv('CLEAN_RELEASE_DIR'), default=True)
RUN_HF_UPLOAD = truthy(os.getenv('RUN_HF_UPLOAD'), default=False)
RUN_GITHUB_RELEASE = truthy(os.getenv('RUN_GITHUB_RELEASE'), default=False)
HF_PRIVATE = truthy(os.getenv('HF_PRIVATE'), default=False)
GITHUB_DRAFT_RELEASE = truthy(os.getenv('GITHUB_DRAFT_RELEASE'), default=True)
GITHUB_PRERELEASE = truthy(os.getenv('GITHUB_PRERELEASE'), default=False)
REPLACE_GITHUB_ASSETS = truthy(os.getenv('REPLACE_GITHUB_ASSETS'), default=False)

if PACKAGING_MODE not in {'hardlink', 'copy'}:
    raise ValueError('PACKAGING_MODE must be hardlink or copy')

print('Runtime:', 'Colab' if IS_COLAB else 'Local')
print('Project root:', PROJECT_ROOT)
print('Quantized dir:', QUANTIZED_DIR)
print('Release root:', RELEASE_ROOT)
print('Release tag:', RELEASE_TAG)
print('Include experimental artifacts:', INCLUDE_EXPERIMENTAL_ARTIFACTS)
print('Include failed SVD artifacts:', INCLUDE_FAILED_SVD_ARTIFACTS)
print('Packaging mode:', PACKAGING_MODE)


In [ ]:
from getpass import getpass
from urllib.parse import urlparse

from dotenv import load_dotenv

ENV_FILE = PROJECT_ROOT / '.env'
if ENV_FILE.exists():
    load_dotenv(ENV_FILE)
    print('Loaded environment from', ENV_FILE)
else:
    print('No .env file found at project root; using process environment and prompts only when upload flags are enabled.')


def parse_github_repo_id(remote_url: str) -> str:
    remote_url = remote_url.strip()
    if not remote_url:
        return ''
    if remote_url.startswith('git@github.com:'):
        repo = remote_url.split(':', 1)[1]
    else:
        parsed = urlparse(remote_url)
        if parsed.netloc.lower() != 'github.com':
            return ''
        repo = parsed.path.lstrip('/')
    if repo.endswith('.git'):
        repo = repo[:-4]
    return repo if repo.count('/') == 1 else ''


def detect_github_repo_id(project_root: Path) -> str:
    try:
        result = run_command(['git', 'remote', 'get-url', 'origin'], cwd=project_root, check=False)
    except FileNotFoundError:
        return ''
    if result.returncode != 0:
        return ''
    return parse_github_repo_id(result.stdout.strip())


HF_TOKEN = os.getenv('HF_TOKEN', '').strip()
HF_REPO_ID = os.getenv('HF_REPO_ID', '').strip()
GH_TOKEN = os.getenv('GH_TOKEN', '').strip()
GH_REPO_ID = os.getenv('GH_REPO_ID', '').strip() or os.getenv('GITHUB_REPOSITORY', '').strip() or detect_github_repo_id(PROJECT_ROOT)

if RUN_HF_UPLOAD:
    if not HF_TOKEN:
        HF_TOKEN = getpass('Hugging Face token with write access: ')
    if not HF_REPO_ID:
        HF_REPO_ID = input('Hugging Face repo ID, for example owner/model-repo: ').strip()

if RUN_GITHUB_RELEASE:
    if not GH_TOKEN:
        GH_TOKEN = getpass('GitHub token with repo release access: ')
    if not GH_REPO_ID:
        GH_REPO_ID = input('GitHub repo ID, for example owner/repo: ').strip()

print('HF_REPO_ID:', HF_REPO_ID or '(not set)')
print('HF_TOKEN configured:', bool(HF_TOKEN))
print('GH_REPO_ID:', GH_REPO_ID or '(not set)')
print('GH_TOKEN configured:', bool(GH_TOKEN))
print('RUN_HF_UPLOAD:', RUN_HF_UPLOAD)
print('RUN_GITHUB_RELEASE:', RUN_GITHUB_RELEASE)


## Artifact catalog

The catalog below names the quantized files that exist in this repository and records their status. The recommended release artifact is `gemma_ternary_aggressive.pt`. The SVD Sub1Bit 90 percent threshold artifacts are included in inventory because they are real files, but they are marked `BROKEN` and skipped by default.


In [ ]:
import hashlib
import json
from typing import Any

ARTIFACT_CATALOG: dict[str, dict[str, Any]] = {
    'gemma_ternary_aggressive.pt': {
        'method': 'Ternary Aggressive',
        'status': 'RECOMMENDED',
        'publish_group': 'primary',
        'format': 'pytorch',
        'reported_metrics': {
            'bpw': 1.60,
            'compression': '10x reported in README.md',
            'estimated_ppl': '~66',
            'average_reconstruction_mse': 0.243,
        },
        'notes': 'Best surviving path in this checkout: direct ternary quantization with importance-weighted bit allocation.',
    },
    'gemma_magq.pt': {
        'method': 'Magnitude Quant (4-bit/2-bit)',
        'status': 'QUALITY_BASELINE',
        'publish_group': 'baseline',
        'format': 'pytorch',
        'reported_metrics': {
            'bpw': 2.13,
            'compression': '7.5x BPW-based estimate in README.md; file-size ratio is lower',
            'estimated_ppl': '~128',
            'average_reconstruction_mse': 0.494,
        },
        'notes': 'Quality baseline with per-channel magnitude scaling; larger than ternary aggressive.',
    },
    'gemma_hybrid_stream.pt': {
        'method': 'Hybrid Stream',
        'status': 'EXPERIMENTAL',
        'publish_group': 'experimental',
        'format': 'pytorch',
        'reported_metrics': {},
        'notes': 'Experimental streamed hybrid artifact. Keep included only when experimental artifacts are wanted.',
    },
    'gemma-4-E2B-sub1bit-stream.gguf': {
        'method': 'SVD Sub1Bit (90% threshold)',
        'status': 'BROKEN',
        'publish_group': 'failed_svd',
        'format': 'gguf-stream',
        'reported_metrics': {
            'bpw': 0.88,
            'compression': '18.1x reported for failed SVD path',
            'average_reconstruction_mse': 1123,
        },
        'notes': 'Readable streaming GGUF export of the failed SVD Sub1Bit experiment; quality is not validated for release use.',
    },
    'gemma-4-E2B-sub1bit.pt': {
        'method': 'SVD Sub1Bit (90% threshold)',
        'status': 'BROKEN',
        'publish_group': 'failed_svd',
        'format': 'pytorch',
        'reported_metrics': {
            'bpw': 0.88,
            'compression': '18.1x reported for failed SVD path',
            'average_reconstruction_mse': 1123,
        },
        'notes': 'Failed SVD Sub1Bit checkpoint. The docs report near-full-rank factors and broken reconstruction.',
    },
    'gemma-4-E2B-sub1bit.gguf': {
        'method': 'SVD Sub1Bit (90% threshold)',
        'status': 'BROKEN',
        'publish_group': 'failed_svd',
        'format': 'gguf',
        'reported_metrics': {
            'bpw': 0.88,
            'compression': '3.6x file-size ratio reported in EVAL_RESULTS.md',
            'average_reconstruction_mse': 1123,
        },
        'notes': 'GGUF export for the failed SVD Sub1Bit path. Publish only when failure artifacts are intentionally archived.',
    },
}

IGNORED_QUANTIZED_FILES = {'test_gguf.bin', 'test_write.bin'}
DOC_SOURCES = ['README.md', 'EVAL_RESULTS.md', 'FINAL_SUMMARY.txt']


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def size_mib(size_bytes: int) -> float:
    return round(size_bytes / (1024 * 1024), 2)


def selected_for_release(meta: dict[str, Any]) -> bool:
    status = meta['status']
    if status == 'BROKEN':
        return INCLUDE_FAILED_SVD_ARTIFACTS
    if status == 'EXPERIMENTAL':
        return INCLUDE_EXPERIMENTAL_ARTIFACTS
    return True


def build_artifact_records(compute_hashes: bool = False) -> tuple[list[dict[str, Any]], list[dict[str, Any]], list[str]]:
    records: list[dict[str, Any]] = []
    missing: list[dict[str, Any]] = []
    unexpected: list[str] = []

    for filename, meta in ARTIFACT_CATALOG.items():
        path = QUANTIZED_DIR / filename
        if not path.exists():
            missing.append({'filename': filename, **meta})
            continue
        stat = path.stat()
        record = {
            'filename': filename,
            'source_repo_path': f'quantized/{filename}',
            'release_repo_path': f'artifacts/{filename}',
            'size_bytes': stat.st_size,
            'size_mib': size_mib(stat.st_size),
            'mtime_utc': datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
            'sha256': sha256_file(path) if compute_hashes else None,
            'selected_for_release': selected_for_release(meta),
            **meta,
        }
        records.append(record)

    for path in sorted(QUANTIZED_DIR.iterdir()):
        if path.is_file() and path.name not in ARTIFACT_CATALOG and path.name not in IGNORED_QUANTIZED_FILES:
            unexpected.append(path.name)

    return records, missing, unexpected


In [ ]:
artifact_records, missing_artifacts, unexpected_artifacts = build_artifact_records(compute_hashes=False)

if not artifact_records:
    raise FileNotFoundError(f'No cataloged quantized artifacts found in {QUANTIZED_DIR}')

print('Cataloged artifacts:')
for record in artifact_records:
    marker = 'include' if record['selected_for_release'] else 'skip'
    print(f" - {marker:7} {record['filename']:34} {record['status']:16} {record['size_mib']:10.2f} MiB  {record['method']}")

if missing_artifacts:
    print('\nMissing catalog entries:')
    for item in missing_artifacts:
        print(' -', item['filename'], item['status'])

if unexpected_artifacts:
    print('\nUnexpected quantized files not packaged by this release flow:')
    for name in unexpected_artifacts:
        print(' -', name)

selected_count = sum(1 for record in artifact_records if record['selected_for_release'])
if selected_count == 0:
    raise RuntimeError('No artifacts selected for release. Check INCLUDE_EXPERIMENTAL_ARTIFACTS and INCLUDE_FAILED_SVD_ARTIFACTS.')
print(f'\nSelected artifacts for release: {selected_count}')


## Stage release folder

This cell creates a reproducible release folder. It stages artifact files under `artifacts/`, writes checksums, emits a Hugging Face model card as `README.md`, and writes a structured `release_manifest.json` with both included and skipped inventory.

When `PACKAGING_MODE=hardlink`, local runs avoid duplicating multi-GB files on the same filesystem. If hardlinking is not possible, the cell falls back to copying. Colab Drive targets usually copy.


In [ ]:
from textwrap import dedent

RELEASE_DIR = (RELEASE_ROOT / RELEASE_NAME / RELEASE_TAG).resolve()
ARTIFACTS_DIR = RELEASE_DIR / 'artifacts'
RELEASE_ROOT.mkdir(parents=True, exist_ok=True)

# Guard against accidental cleanup outside the configured release root.
resolved_root = RELEASE_ROOT.resolve()
if resolved_root not in RELEASE_DIR.parents and RELEASE_DIR != resolved_root:
    raise RuntimeError(f'Release directory {RELEASE_DIR} is not under release root {resolved_root}')

if CLEAN_RELEASE_DIR and RELEASE_DIR.exists():
    shutil.rmtree(RELEASE_DIR)

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def stage_file(source: Path, destination: Path) -> str:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    if PACKAGING_MODE == 'copy':
        shutil.copy2(source, destination)
        return 'copy'
    try:
        os.link(source, destination)
        return 'hardlink'
    except OSError:
        shutil.copy2(source, destination)
        return 'copy-fallback'


all_records, missing_records, unexpected_files = build_artifact_records(compute_hashes=True)
release_records: list[dict[str, Any]] = []
skipped_records: list[dict[str, Any]] = []

for record in all_records:
    if record['selected_for_release']:
        source = PROJECT_ROOT / record['source_repo_path']
        destination = RELEASE_DIR / record['release_repo_path']
        staging_method = stage_file(source, destination)
        record = {**record, 'staging_method': staging_method}
        release_records.append(record)
    else:
        skipped_records.append(record)

if not release_records:
    raise RuntimeError('No artifacts were staged for release.')

created_at_utc = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
manifest = {
    'schema_version': 1,
    'project': 'sub1quant',
    'release_name': RELEASE_NAME,
    'release_tag': RELEASE_TAG,
    'created_at_utc': created_at_utc,
    'base_model': 'Gemma 4 E2B from local models/gemma-4-E2B checkout',
    'repository': 'https://github.com/toxzak-svg/Quantization-Exploration',
    'artifact_policy': {
        'include_experimental_artifacts': INCLUDE_EXPERIMENTAL_ARTIFACTS,
        'include_failed_svd_artifacts': INCLUDE_FAILED_SVD_ARTIFACTS,
        'packaging_mode': PACKAGING_MODE,
    },
    'source_documents': DOC_SOURCES,
    'summary': {
        'recommended_artifact': 'gemma_ternary_aggressive.pt',
        'recommended_method': 'Ternary Aggressive',
        'reported_bpw': 1.60,
        'reported_compression': '10x',
        'reported_estimated_ppl': '~66',
        'sub_1bit_target_status': 'Not met by recommended artifact; failed SVD artifact is below 1 BPW but broken.',
    },
    'artifacts': release_records,
    'skipped_artifacts': skipped_records,
    'missing_catalog_artifacts': missing_records,
    'unexpected_quantized_files': unexpected_files,
}


def artifact_table(records: list[dict[str, Any]]) -> str:
    rows = ['| File | Status | Method | Size | SHA-256 |', '|---|---:|---|---:|---|']
    for record in records:
        rows.append(
            f"| `{record['filename']}` | {record['status']} | {record['method']} | "
            f"{record['size_mib']:.2f} MiB | `{record['sha256']}` |"
        )
    return '\n'.join(rows)


def skipped_table(records: list[dict[str, Any]]) -> str:
    if not records:
        return 'No cataloged artifacts were skipped.'
    rows = ['| File | Status | Reason |', '|---|---:|---|']
    for record in records:
        reason = 'Set INCLUDE_FAILED_SVD_ARTIFACTS=1 to include this failed SVD artifact.' if record['status'] == 'BROKEN' else 'Selection flag disabled.'
        rows.append(f"| `{record['filename']}` | {record['status']} | {reason} |")
    return '\n'.join(rows)


model_card = f"""---
library_name: pytorch
tags:
- quantization
- ternary
- low-bit
- gemma
- research-artifact
pipeline_tag: text-generation
---

# sub1quant Gemma 4 E2B quantized artifacts

This repository contains quantized artifacts produced by the `sub1quant` experiments for Gemma 4 E2B. The release is intentionally explicit about what worked and what did not.

## Recommended artifact

`gemma_ternary_aggressive.pt` is the recommended artifact in this checkout.

- Method: Ternary Aggressive
- Status: RECOMMENDED
- Reported BPW: 1.60
- Reported compression: 10x
- Reported estimated perplexity: ~66
- Reported average reconstruction MSE: 0.243

This does not meet the original sub-1-bit target, but it is the best surviving path documented by `README.md` and `EVAL_RESULTS.md`.

## Important limitation

The SVD Sub1Bit (90% threshold) artifacts are real files in the repository, but the project docs mark that path as BROKEN. The reported failure mode is near-full-rank SVD factors plus ternary quantization, producing broken reconstruction with average MSE around 1123. These files are skipped by default unless `INCLUDE_FAILED_SVD_ARTIFACTS=1` is set.

## Included files

{artifact_table(release_records)}

## Skipped cataloged files

{skipped_table(skipped_records)}

## Manifest

See `release_manifest.json` for machine-readable metadata, checksums, source document references, and selection flags.

## Reproduce this release

Run `notebook/colab_hf_github_pipeline.ipynb` from a local checkout or from Colab with the checkout and `quantized/` folder mounted. Set `RUN_HF_UPLOAD=1` for Hugging Face upload and `RUN_GITHUB_RELEASE=1` for GitHub Release asset upload.

## License and redistribution

This release does not assert a new model license. Verify the base model license, local repository license, and redistribution permissions before making a public release.
"""

checksums = ''.join(f"{record['sha256']}  artifacts/{record['filename']}\n" for record in release_records)
github_release_notes = f"""# {RELEASE_TAG}

Release folder generated at {created_at_utc}.

Recommended artifact: `gemma_ternary_aggressive.pt` using Ternary Aggressive quantization.

The SVD Sub1Bit (90% threshold) path is documented as BROKEN and is skipped by default. See `release_manifest.json` for full inventory, checksums, and selected/skipped status.
"""

gitattributes = """*.pt filter=lfs diff=lfs merge=lfs -text
*.gguf filter=lfs diff=lfs merge=lfs -text
*.bin filter=lfs diff=lfs merge=lfs -text
"""

(RELEASE_DIR / 'release_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
(RELEASE_DIR / 'README.md').write_text(model_card, encoding='utf-8')
(RELEASE_DIR / 'checksums.sha256').write_text(checksums, encoding='utf-8')
(RELEASE_DIR / 'github_release_notes.md').write_text(github_release_notes, encoding='utf-8')
(RELEASE_DIR / '.gitattributes').write_text(gitattributes, encoding='utf-8')

print('Release directory:', RELEASE_DIR)
print('Staged artifacts:')
for record in release_records:
    print(f" - {record['filename']} ({record['size_mib']:.2f} MiB, {record['staging_method']})")
print('Wrote release_manifest.json, README.md, checksums.sha256, github_release_notes.md, and .gitattributes')


In [ ]:
print('Release tree:')
for path in sorted(RELEASE_DIR.rglob('*')):
    if path.is_file():
        rel = path.relative_to(RELEASE_DIR).as_posix()
        print(f" - {rel:55} {path.stat().st_size / (1024 * 1024):10.2f} MiB")

manifest_preview = json.loads((RELEASE_DIR / 'release_manifest.json').read_text(encoding='utf-8'))
print('\nManifest summary:')
print(json.dumps({
    'release_tag': manifest_preview['release_tag'],
    'recommended_artifact': manifest_preview['summary']['recommended_artifact'],
    'artifact_count': len(manifest_preview['artifacts']),
    'skipped_count': len(manifest_preview['skipped_artifacts']),
}, indent=2))


## Publish to Hugging Face Hub

This cell is a dry run unless `RUN_HF_UPLOAD=1` is set. It creates the model repository if needed and uploads the staged release folder with `upload_folder`, preserving the manifest, model card, checksums, and artifact files.


In [ ]:
from huggingface_hub import HfApi, upload_folder

if RUN_HF_UPLOAD:
    if not HF_TOKEN:
        raise ValueError('HF_TOKEN is required when RUN_HF_UPLOAD=1')
    if not HF_REPO_ID:
        raise ValueError('HF_REPO_ID is required when RUN_HF_UPLOAD=1')

    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO_ID, repo_type='model', private=HF_PRIVATE, exist_ok=True)
    print(f'Uploading {RELEASE_DIR} to Hugging Face model repo {HF_REPO_ID}')
    upload_folder(
        folder_path=str(RELEASE_DIR),
        repo_id=HF_REPO_ID,
        repo_type='model',
        token=HF_TOKEN,
        commit_message=f'Release {RELEASE_TAG}',
    )
    print('Hugging Face upload complete:', HF_REPO_ID)
else:
    print('Dry run: set RUN_HF_UPLOAD=1, HF_TOKEN, and HF_REPO_ID to upload this release folder.')
    print('Release folder ready for HF upload:', RELEASE_DIR)


## Publish to GitHub Releases

This cell is also a dry run unless `RUN_GITHUB_RELEASE=1` is set. It uses the GitHub REST API to create or reuse a release tag and uploads the staged files as release assets. Large `.pt` and `.gguf` files should go to GitHub Releases or Git LFS, not ordinary Git history.


In [ ]:
import mimetypes

import requests

GITHUB_API_ROOT = 'https://api.github.com'


def github_headers(extra: dict[str, str] | None = None) -> dict[str, str]:
    if not GH_TOKEN:
        raise ValueError('GH_TOKEN is required for GitHub API calls')
    headers = {
        'Accept': 'application/vnd.github+json',
        'Authorization': f'Bearer {GH_TOKEN}',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    if extra:
        headers.update(extra)
    return headers


def github_request(method: str, url: str, **kwargs) -> requests.Response:
    response = requests.request(method, url, headers=github_headers(kwargs.pop('headers', None)), timeout=120, **kwargs)
    if response.status_code >= 400:
        raise RuntimeError(f'GitHub API {method} {url} failed: {response.status_code} {response.text[:500]}')
    return response


def create_github_release(repo_id: str, tag_name: str, release_name: str, body: str) -> dict[str, Any]:
    existing_url = f'{GITHUB_API_ROOT}/repos/{repo_id}/releases/tags/{tag_name}'
    existing = requests.get(existing_url, headers=github_headers(), timeout=120)
    if existing.status_code == 200:
        print('Using existing GitHub release:', tag_name)
        return existing.json()
    if existing.status_code != 404:
        raise RuntimeError(f'GitHub release lookup failed: {existing.status_code} {existing.text[:500]}')

    create_url = f'{GITHUB_API_ROOT}/repos/{repo_id}/releases'
    payload = {
        'tag_name': tag_name,
        'name': release_name,
        'body': body,
        'draft': GITHUB_DRAFT_RELEASE,
        'prerelease': GITHUB_PRERELEASE,
    }
    response = github_request('POST', create_url, json=payload)
    print('Created GitHub release:', tag_name)
    return response.json()


def list_release_assets(release: dict[str, Any]) -> dict[str, dict[str, Any]]:
    response = github_request('GET', release['assets_url'])
    return {asset['name']: asset for asset in response.json()}


def upload_release_asset(release: dict[str, Any], file_path: Path, replace_existing: bool = False) -> None:
    assets = list_release_assets(release)
    if file_path.name in assets:
        if not replace_existing:
            print('Skipping existing GitHub asset:', file_path.name)
            return
        github_request('DELETE', assets[file_path.name]['url'])
        print('Deleted existing GitHub asset:', file_path.name)

    upload_url = release['upload_url'].split('{', 1)[0]
    content_type = mimetypes.guess_type(file_path.name)[0] or 'application/octet-stream'
    params = {'name': file_path.name}
    headers = github_headers({'Content-Type': content_type})
    with file_path.open('rb') as handle:
        response = requests.post(upload_url, headers=headers, params=params, data=handle, timeout=None)
    if response.status_code >= 400:
        raise RuntimeError(f'GitHub asset upload failed for {file_path.name}: {response.status_code} {response.text[:500]}')
    print('Uploaded GitHub asset:', file_path.name)


release_asset_paths = [
    RELEASE_DIR / 'README.md',
    RELEASE_DIR / 'release_manifest.json',
    RELEASE_DIR / 'checksums.sha256',
    RELEASE_DIR / 'github_release_notes.md',
    *sorted(ARTIFACTS_DIR.iterdir()),
]

if RUN_GITHUB_RELEASE:
    if not GH_TOKEN:
        raise ValueError('GH_TOKEN is required when RUN_GITHUB_RELEASE=1')
    if not GH_REPO_ID:
        raise ValueError('GH_REPO_ID is required when RUN_GITHUB_RELEASE=1')

    notes = (RELEASE_DIR / 'github_release_notes.md').read_text(encoding='utf-8')
    release = create_github_release(GH_REPO_ID, RELEASE_TAG, RELEASE_TAG, notes)
    for file_path in release_asset_paths:
        upload_release_asset(release, file_path, replace_existing=REPLACE_GITHUB_ASSETS)
    print('GitHub release upload complete:', f'https://github.com/{GH_REPO_ID}/releases/tag/{RELEASE_TAG}')
else:
    print('Dry run: set RUN_GITHUB_RELEASE=1, GH_TOKEN, and GH_REPO_ID to create/upload a GitHub Release.')
    print('Release assets that would be uploaded:')
    for file_path in release_asset_paths:
        print(' -', file_path)


## Local command runbook

For a local non-notebook run, set the same environment variables and execute the cells top to bottom. For Colab, mount Drive, ensure this checkout plus `quantized/` is visible, then execute top to bottom.

Minimal publish sequence:

```bash
# local shell or Colab environment
export HF_REPO_ID=owner/sub1quant-gemma-4-e2b
export HF_TOKEN=hf_...
export GH_REPO_ID=owner/repo
export GH_TOKEN=ghp_...
export RELEASE_TAG=sub1quant-gemma-4-e2b-ternary-aggressive-v1
export RUN_HF_UPLOAD=1
export RUN_GITHUB_RELEASE=1
```

Keep `INCLUDE_FAILED_SVD_ARTIFACTS=0` for a normal release. Set it to `1` only for an archival release that intentionally publishes the broken SVD Sub1Bit artifacts with failure metadata.
